In [2]:
cd ..

/Users/harini/Dropbox (MIT)/Harini Narayanan’s files/Home/CodonOptimization/Models/BO_forHyperParameter


In [3]:
import math
import collections
import pickle
import random
%matplotlib inline
from matplotlib import pyplot as plt

import GPy
from GPy import likelihoods
import numpy as np
import pandas as pd
from tqdm import tqdm

# from Kernel import MixtureViaSumAndProduct, CategoryOverlapKernel
from InitialData_Gen import initialize

# from SamplingCategorical import compute_prob_dist_and_draw_hts
from UpdateCategoricalWeight import compute_reward_for_all_cat_variable, update_weights_for_all_cat_var

from AskTell import ask_tell

from scipy.optimize import minimize

from typing import Union, Tuple
from paramz.transformations import Logexp
import scipy
from scipy.optimize import minimize
from scipy.optimize import NonlinearConstraint



# np.random.seed(37)

In [4]:
bounds = [  {'name': 'Enc hidden size', 'type': 'continuous', 'domain': (60, 515)},
            {'name': 'Enc Embedding size', 'type': 'continuous', 'domain': (30, 260)},
         {'name': 'Dec Embedding size', 'type': 'continuous', 'domain': (30, 260)},
         {'name': 'Dense Layer size', 'type': 'continuous', 'domain': (30, 260)},
         {'name': 'Drop rate', 'type': 'continuous', 'domain': (0, 0.9)},]
    
Nx = len(bounds)
initN = 12
Niter = 1
batch_size = 3
approach_type = 'Co'
prob_type = 'UnConstrained'

data_param = {'Nx': Nx, 'nDim': Nx, 'bounds': bounds, 
              'approach_type': approach_type,
              'prob_type': prob_type, 'initN': initN}

In [8]:
runs0 = [0, 1, 2, 3, 4, 5, 6, 8, 9, 10 ,11]
data0 = pd.read_csv('../BO_forHyperParameter/Arch1/InitialRound_HyperParameter.csv').iloc[runs0,1:].values
result0 = np.zeros((len(runs0), 1))

for i in range(len(runs0)):
    filename = '../EncDec/Loss_Evolution/Initial/Combo' + str(runs0[i]) + '.csv'
    output = pd.read_csv(filename).iloc[:,1:].values
    result0[i] = output[-1,-1]
    
runs1 = [0, 1, 2]
data1 = pd.read_csv('../BO_forHyperParameter/Arch1/Round1.csv').iloc[runs1,1:].values
result1 = np.zeros((len(runs1), 1))

for i in range(len(runs1)):
    filename = '../EncDec/Loss_Evolution/Round1/Combo' + str(runs1[i]) + '.csv'
    output = pd.read_csv(filename).iloc[:,1:].values
    result1[i] = output[-1,-1]
    
runs2 = [0, 1, 2]
data2 = pd.read_csv('../BO_forHyperParameter/Arch1/Round2.csv').iloc[runs2,1:].values
result2 = np.zeros((len(runs2), 1))

for i in range(len(runs2)):
    filename = '../EncDec/Loss_Evolution/Round2/Combo' + str(runs2[i]) + '.csv'
    output = pd.read_csv(filename).iloc[:,1:].values
    result2[i] = output[-1,-1]

runs3 = [0, 1, 2]
data3 = pd.read_csv('../BO_forHyperParameter/Arch1/Round3.csv').iloc[runs3,1:].values
result3 = np.zeros((len(runs3), 1))

for i in range(len(runs3)):
    filename = '../EncDec/Loss_Evolution/Round3/Combo' + str(runs3[i]) + '.csv'
    output = pd.read_csv(filename).iloc[:,1:].values
    result3[i] = output[-1,-1]
    
data = np.concatenate((data0, data1, data2, data3), axis = 0)
result = np.concatenate((result0, result1, result2, result3), axis = 0)

FileNotFoundError: [Errno 2] No such file or directory: '../EncDec/Loss_Evolution/Round3/Combo1.csv'

In [ ]:
Wc_list = []
gamma_list = []
    
# for i in range(Niter):
X_exp, f_val, gp = ask_tell(data, result,data_param,
                        'Matern32', 'thompson_sampling', batch_size,  Wc_list, gamma_list) #thompson_sampling

X_ts_norm = (X_exp - np.mean(data, 0))/np.std(data, 0)
print(X_ts_norm)
Yp = gp.predict(X_ts_norm)
print(Yp)

In [ ]:
column_names = []
for i in range(len(bounds)):
    column_names.append(data_param['bounds'][i]['name'])
    

X_exp[:,0:-1] = np.round(X_exp[:,0:-1])
X_exp[:,-1] = np.round(X_exp[:,-1],1)
X_exp_df = pd.DataFrame(X_exp)
X_exp_df.columns = column_names


pd.DataFrame(X_exp_df).to_csv('Round3.csv')

In [ ]:
column_names